In [ ]:
!uv sync --quiet

import torch
import math
import timeit
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import gc

from pytorch_semifield_conv import SelectSemifield, QuadraticKernelSpectral2D

In [ ]:
# Move to project root
from pathlib import Path
import os

if not Path("./src").is_dir():
    for parent_path in Path.cwd().parents:
        if (parent_path / "src").is_dir():
            os.chdir(parent_path)
            break
    else:
        raise FileNotFoundError("Can't find project root")

assert Path("./src").is_dir()

In [ ]:
from src import load_data

In [ ]:
torch.manual_seed(0)
kmnist_data = load_data.k_mnist().x_train.cuda()
rand_data_small = torch.rand((2048, 20, 12, 12)).cuda()
rand_data_med = torch.rand((2048, 32, 32, 32)).cuda()
rand_data_large = torch.rand((2048, 50, 128, 128)).cuda()

In [ ]:
def time_conv(
        conv,
        imgs: torch.Tensor,
        kernels: torch.Tensor,
        target_time: float = 0.2,
        cudagraphs: bool = True,
        **kwargs,
):
    imgs, kernels = imgs.cuda(), kernels.cuda()

    gc.collect()
    torch.cuda.empty_cache()

    if cudagraphs:
        torch.compiler.cudagraph_mark_step_begin()
        conv = torch.compile(conv, mode="reduce-overhead", fullgraph=True)

    conv(imgs, kernels, **kwargs)

    extra_kwargs = ", ".join(f"{name}={val}" for name, val in kwargs.items())
    code = f"""
conv(imgs, kernels, {extra_kwargs})
sync()
"""
    timer = timeit.Timer(
        code,
        globals={
            "conv": conv,
            "imgs": imgs,
            "kernels": kernels,
            "sync": torch.cuda.synchronize,
        },
    )
    timer.timeit(2)
    autorange_runs, autorange_time = timer.autorange()
    target_runs = math.ceil(autorange_runs / autorange_time * target_time / 2)

    # Run it twice to reduce the effects of interference
    return min(timer.repeat(2, target_runs)) / target_runs

In [ ]:
inflate_sizes = (1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 92, 128)

In [ ]:
def inflation_timings(imgs: torch.Tensor, kernel_size: int, disable_bar: bool = False, **kwargs):
    channels = imgs.shape[1]
    torch.manual_seed(0)
    kernels = QuadraticKernelSpectral2D(channels, channels, kernel_size)().cuda()
    if kernel_size % 2:
        padding = kernel_size // 2
    else:
        padding = (kernel_size // 2 - 1, kernel_size // 2)

    return [
        time_conv(
            SelectSemifield.tropical_max().lazy_fixed(kernel_inflation=size, to_extension=False),
            imgs,
            kernels,
            padding=padding,
            groups=channels,
            **kwargs,
        )
        for size in tqdm(inflate_sizes, disable=disable_bar, desc="Inflation sizes")
    ]

In [ ]:
test_timings = inflation_timings(kmnist_data[:1024], 5)
print(test_timings)

In [ ]:
def inflation_plot(inflation_data: list[float] | dict[str, list[float]], title: str, ax: plt.Axes = None):
    if isinstance(inflation_data, list):
        inflation_data = {"": inflation_data}

    if ax is None:
        _, ax = plt.subplots(layout="compressed")

    ax.set_title(title)
    for label, data in inflation_data.items():
        color = ax.plot(inflate_sizes, data, label=label, linestyle="dashed")[0].get_color()
        ax.scatter(inflate_sizes, data, color=color)
        best = np.argmin(data)
        ax.scatter(inflate_sizes[best], data[best], marker="x")

    ax.legend()
    ax.set_xscale("log", base=2)
    ax.set_yscale("log", base=10)
    ax.set_xlabel("Kernel inflation coefficient in backwards pass")
    ax.set_ylabel("Runtime in seconds")

In [ ]:
inflation_plot(test_timings, "$5\\times5$ kernel on K-MNIST, batch size 1024")

In [ ]:
batch_sizes = np.logspace(2, 10, num=9, base=2).astype(int).tolist()
print(batch_sizes)